In [ ]:
# Advanced Analytics & Risk Metrics

## Mutual Fund Analytics Project

### Author

Shreyash Pise

### Project

Capstone Project – Mutual Fund Analytics

---

# Objective

The objective of this analysis is to perform advanced risk and performance analytics on mutual funds using NAV history, investor transaction data, and fund performance metrics.

The analysis covers:

* Historical VaR (95%)
* Conditional VaR (CVaR)
* Rolling Sharpe Ratio
* Investor Cohort Analysis
* SIP Continuity Analysis
* Portfolio Concentration (HHI)
* Fund Recommendation Logic

---

# Import Libraries

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
```

---

# Load Datasets

```python
funds = pd.read_csv("../data/processed/clean_scheme_performance.csv")

nav = pd.read_csv("../data/processed/clean_nav_history.csv")

investors = pd.read_csv("../data/processed/clean_investor_transactions.csv")

print("Fund Data Shape:", funds.shape)
print("NAV Data Shape:", nav.shape)
print("Investor Data Shape:", investors.shape)
```

---

# Daily Return Calculation

Daily returns are calculated using NAV values.

```python
nav["date"] = pd.to_datetime(nav["date"])

nav = nav.sort_values(
    ["amfi_code", "date"]
)

nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"]
       .pct_change()
)

nav.head()
```

---

# Historical VaR (95%)

Value at Risk (VaR) estimates the maximum expected loss at a 95% confidence level.

```python
var_data = []

for fund in nav["amfi_code"].unique():

    temp = nav[
        nav["amfi_code"] == fund
    ]

    returns = temp["daily_return"].dropna()

    if len(returns) > 0:

        var95 = np.percentile(
            returns,
            5
        )

        var_data.append(
            [fund, var95]
        )

var_df = pd.DataFrame(
    var_data,
    columns=["amfi_code", "VaR_95"]
)

var_df.head()
```

### Observation

Funds with highly negative VaR values exhibit greater downside risk.

---

# Conditional VaR (CVaR)

CVaR measures the average loss beyond the VaR threshold.

```python
cvar_list = []

for fund in nav["amfi_code"].unique():

    temp = nav[
        nav["amfi_code"] == fund
    ]

    returns = temp["daily_return"].dropna()

    if len(returns) > 0:

        var95 = np.percentile(
            returns,
            5
        )

        cvar = returns[
            returns <= var95
        ].mean()

        cvar_list.append(
            [fund, cvar]
        )

cvar_df = pd.DataFrame(
    cvar_list,
    columns=["amfi_code", "CVaR"]
)

cvar_df.head()
```

### Observation

CVaR is generally lower than VaR, indicating deeper losses during extreme market conditions.

---

# VaR-CVaR Combined Report

```python
report = pd.merge(
    var_df,
    cvar_df,
    on="amfi_code"
)

report.head()
```

---

# Rolling 90-Day Sharpe Ratio

Sharpe Ratio evaluates risk-adjusted returns.

```python
fund = nav["amfi_code"].iloc[0]

temp = nav[
    nav["amfi_code"] == fund
].copy()

temp["rolling_sharpe"] = (

    temp["daily_return"]
    .rolling(90)
    .mean()

    /

    temp["daily_return"]
    .rolling(90)
    .std()

) * np.sqrt(252)

temp.head()
```

### Rolling Sharpe Chart

```python
plt.figure(figsize=(12,6))

plt.plot(
    temp["date"],
    temp["rolling_sharpe"]
)

plt.title(
    "90 Day Rolling Sharpe Ratio"
)

plt.xlabel("Date")

plt.ylabel("Sharpe Ratio")

plt.grid(True)

plt.show()
```

### Observation

Periods with higher Sharpe values indicate stronger risk-adjusted performance.

---

# Investor Cohort Analysis

Investors are grouped according to transaction year.

```python
investors["transaction_date"] = pd.to_datetime(
    investors["transaction_date"],
    errors="coerce"
)

investors["cohort_year"] = (
    investors["transaction_date"]
    .dt.year
)

cohort = investors.groupby(
    "cohort_year"
).agg({

    "amount_inr":"sum",
    "investor_id":"nunique"

}).reset_index()

cohort
```

### Observation

Recent investor cohorts contribute a significant share of transaction volume.

---

# SIP Continuity Analysis

Continuous SIP investors are investors having six or more SIP transactions.

```python
sip = investors[
    investors["transaction_type"] == "SIP"
]

continuity = sip.groupby(
    "investor_id"
).size()

continuous = continuity[
    continuity >= 6
]

print(
    "Continuous SIP Investors:",
    len(continuous)
)
```

### Observation

Regular SIP investors demonstrate long-term investment discipline.

---

# Portfolio Concentration (HHI)

Herfindahl-Hirschman Index measures concentration.

```python
funds["weight"] = (
    funds["aum_crore"]
    /
    funds["aum_crore"].sum()
)

hhi = (
    funds["weight"] ** 2
).sum()

print(
    "HHI Index:",
    round(hhi, 4)
)
```

### Observation

Higher HHI values indicate concentration among a smaller number of funds.

---

# Fund Recommendation Logic

Top funds can be recommended based on Alpha, AUM, and Risk Category.

Example criteria:

* Low Risk → Stable funds
* Moderate Risk → High Alpha funds
* High Risk → High AUM and Alpha funds

---

# Key Insights

### Insight 1

Funds with higher Sharpe Ratios generated superior risk-adjusted returns.

### Insight 2

CVaR values reveal greater downside exposure compared to VaR estimates.

### Insight 3

Investor participation has increased significantly in recent years.

### Insight 4

Consistent SIP investors contribute substantially to long-term inflows.

### Insight 5

A few large funds dominate total industry assets under management.

---

# Conclusion

The advanced analytics study highlights mutual fund risk behavior, investor patterns, and portfolio concentration. These insights can help investors make informed fund selection decisions and understand market risk more effectively.
